# How every number on our dashboard is calculated — by hand

**22AIE301 Probabilistic Reasoning** · Sai Reddy .A · Rohit Vardhan .M · Jithin Reddy .K · Sai Vandith

This notebook answers three questions:

1. **Where does the data come from?** — the real web request and the raw numbers it returned
2. **How is a percentage like "8%" calculated?** — by hand, with the actual tables, then checked against the software
3. **Why is this probabilistic reasoning and not machine learning?**

Every calculation below is done twice: once **by hand** (plain multiplication and
addition), and once by the **pgmpy library**. If they match, nothing is hidden.

> Scenario mode and the sandbox use exactly this calculation. Live mode adds extra
> steps (soft evidence and a memory filter) on top of it.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
ROOT = here if (here / "src").exists() else here.parent
sys.path.insert(0, str(ROOT))

from src.learn import load_model
from src.engine import RiskEngine
from src.config import STATES, BINS
from src.discretize import bin_value

model  = load_model()          # the same tables scenario mode and the sandbox use
engine = RiskEngine(model)
table  = model.get_cpds        # table("Flood") gives the Flood CPT
print("loaded", model.number_of_nodes(), "nodes")

loaded 18 nodes


---
## Part 1 — Where the data comes from

We do **not** train on a ready-made dataset. The code sends a web request to a
free public weather service and gets back raw hourly numbers.

This is the actual response we saved for **Idukki** (so it works without internet).

In [2]:
from datetime import datetime
from src.feeds import _trailing, districts

row  = districts().set_index("district").loc["Idukki"]
snap = json.loads((ROOT / "artifacts/live_snapshot.json").read_text(encoding="utf-8"))
now  = datetime.fromisoformat(next(f for f in snap["feeds"] if f["district"] == "Idukki")["fetched_at"])
raw  = json.loads((ROOT / f"artifacts/feed_cache/weather_{row.lat:.3f}_{row.lon:.3f}.json").read_text(encoding="utf-8"))

print("REQUEST SENT:")
print(f"  https://api.open-meteo.com/v1/forecast?latitude={row.lat}&longitude={row.lon}")
print("     &hourly=precipitation,...&models=gfs_seamless,ecmwf_ifs025,icon_seamless&past_days=2")
print()
hours, _ = _trailing(raw["hourly"]["time"], raw["hourly"]["precipitation_ecmwf_ifs025"], 24, now)
print("RAW HOURLY RAIN RETURNED (mm, ECMWF model, last 24 h):")
print(" ", hours)
print(f"  ({len(hours)} hours reported; missing hours are simply not counted)")
total = sum(hours)
edges, labels = BINS["rainfall_mm"]
print()
print(f"24-HOUR TOTAL = {total:.2f} mm")
print(f"IMD rule: under {edges[0]} mm = Low, {edges[0]}-{edges[1]} = Moderate, over {edges[1]} = Heavy")
print(f"--> the model receives:  Rainfall = {bin_value('rainfall_mm', total)}")

REQUEST SENT:


  https://api.open-meteo.com/v1/forecast?latitude=9.8497&longitude=76.9681
     &hourly=precipitation,...&models=gfs_seamless,ecmwf_ifs025,icon_seamless&past_days=2

RAW HOURLY RAIN RETURNED (mm, ECMWF model, last 24 h):
  [0.2, 0.2, 0.2, 0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.4, 0.4, 0.4, 1.4, 1.4, 1.4, 1.1, 1.1, 1.1, 0.1, 0.1]
  (20 hours reported; missing hours are simply not counted)

24-HOUR TOTAL = 10.40 mm
IMD rule: under 15.6 mm = Low, 15.6-64.5 = Moderate, over 64.5 = Heavy
--> the model receives:  Rainfall = Low


So the chain is: **web request → raw hourly numbers → add up 24 hours → put into
the official IMD band → give that band to the network.** The network only ever
sees the band ("Low"), never the raw number.

---
## Part 2 — The model is just tables of probabilities

A Bayesian network is a set of tables, one per node. Each table says *"given the
causes, how likely is each outcome"*. Here are the three tables we need for the
Coimbatore example below.

In [3]:
import pandas as pd

soil = pd.DataFrame({s: [table("SoilMoisture").get_value(SoilMoisture=s, Rainfall="Moderate", Humidity=h)
                         for h in STATES["Humidity"]] for s in STATES["SoilMoisture"]},
                    index=[f"rain=Moderate, humidity={h}" for h in STATES["Humidity"]]).round(4)
print("TABLE 1:  P(SoilMoisture | Rainfall, Humidity)")
display(soil)

TABLE 1:  P(SoilMoisture | Rainfall, Humidity)


,Low,Medium,High
"rain=Moderate, humidity=Low",0.5741,0.4047,0.0212
"rain=Moderate, humidity=Medium",0.1748,0.7735,0.0517
"rain=Moderate, humidity=High",0.0400,0.5522,0.4078


In [4]:
river = pd.DataFrame({r: [table("RiverLevel").get_value(RiverLevel=r, Rainfall="Moderate", SoilMoisture=s)
                          for s in STATES["SoilMoisture"]] for r in STATES["RiverLevel"]},
                     index=[f"rain=Moderate, soil={s}" for s in STATES["SoilMoisture"]]).round(4)
print("TABLE 2:  P(RiverLevel | Rainfall, SoilMoisture)")
display(river)

TABLE 2:  P(RiverLevel | Rainfall, SoilMoisture)


,Low,Medium,High
"rain=Moderate, soil=Low",0.9656,0.0279,0.0065
"rain=Moderate, soil=Medium",0.8370,0.1559,0.0071
"rain=Moderate, soil=High",0.3024,0.4997,0.1979


In [5]:
slide = pd.DataFrame({"P(Landslide = Yes)": [table("Landslide").get_value(Landslide="Yes", Rainfall="Moderate",
                                              SoilMoisture=s, Slope="Moderate") for s in STATES["SoilMoisture"]]},
                     index=[f"rain=Moderate, slope=Moderate, soil={s}" for s in STATES["SoilMoisture"]]).round(4)
print("TABLE 3:  P(Landslide | Rainfall, SoilMoisture, Slope)")
display(slide)

TABLE 3:  P(Landslide | Rainfall, SoilMoisture, Slope)


,P(Landslide = Yes)
"rain=Moderate, slope=Moderate, soil=Low",0.0190
"rain=Moderate, slope=Moderate, soil=Medium",0.0900
"rain=Moderate, slope=Moderate, soil=High",0.3512


---
## Part 3 — Calculating "Landslide 8.0%" for Coimbatore, by hand

On the dashboard (scenario: *ordinary active monsoon day*), Coimbatore shows
**Landslide 8.0%**. The readings were: rain **Moderate**, humidity **Medium**,
slope **Moderate**, river gauge **Low**.

The problem: landslide depends on **soil moisture**, and we have **no soil sensor**.
So we have to reason about the soil first.

### Step 1 — first guess about the soil, from rain and humidity

In [6]:
prior = {s: table("SoilMoisture").get_value(SoilMoisture=s, Rainfall="Moderate", Humidity="Medium")
         for s in STATES["SoilMoisture"]}
for s, p in prior.items():
    print(f"P(soil = {s:<6} | rain=Moderate, humidity=Medium) = {p:.4f}")

P(soil = Low    | rain=Moderate, humidity=Medium) = 0.1748
P(soil = Medium | rain=Moderate, humidity=Medium) = 0.7735
P(soil = High   | rain=Moderate, humidity=Medium) = 0.0517


### Step 2 — the river gauge says "Low". What does that tell us about the soil?

This is **reasoning backwards**, from an effect (the river) to a hidden cause (the
soil). We use **Bayes' rule**:

$$P(\text{soil} \mid \text{river}) = \frac{P(\text{soil}) \times P(\text{river} \mid \text{soil})}{\text{sum of that over all soil states}}$$

In [7]:
likelihood = {s: table("RiverLevel").get_value(RiverLevel="Low", Rainfall="Moderate", SoilMoisture=s)
              for s in STATES["SoilMoisture"]}
top   = {s: prior[s] * likelihood[s] for s in prior}
total = sum(top.values())
post  = {s: top[s] / total for s in top}

print(f"{'soil':<8}{'first guess':>12}{'x P(river=Low|soil)':>22}{'= product':>12}{'/ total':>10}")
for s in prior:
    print(f"{s:<8}{prior[s]:>12.4f}{likelihood[s]:>22.4f}{top[s]:>12.4f}{post[s]:>10.4f}")
print(f"{'':<8}{'':>12}{'':>22}{'total = ' + format(total, '.4f'):>12}")

soil     first guess   x P(river=Low|soil)   = product   / total
Low           0.1748                0.9656      0.1688    0.2029
Medium        0.7735                0.8370      0.6475    0.7783
High          0.0517                0.3024      0.0156    0.0188
                                          total = 0.8319


A low river is more likely when the soil is dry, so the belief **shifts towards dry
soil**. No sensor touched the soil — the network worked it out.

### Step 3 — combine the soil belief with the landslide table (add up the cases)

$$P(\text{landslide}) = \sum_{\text{soil}} P(\text{soil} \mid \text{readings}) \times P(\text{landslide} \mid \text{rain, soil, slope})$$

In [8]:
answer = 0.0
for s in STATES["SoilMoisture"]:
    p_slide = table("Landslide").get_value(Landslide="Yes", Rainfall="Moderate", SoilMoisture=s, Slope="Moderate")
    answer += post[s] * p_slide
    print(f"soil={s:<6}:  {post[s]:.4f} x {p_slide:.4f} = {post[s]*p_slide:.4f}")

coimbatore = {'Rainfall': 'Moderate', 'Temperature': 'Normal', 'Humidity': 'Medium', 'WindSpeed': 'Calm',
              'SeaSurfaceTemp': 'Warm', 'PressureDrop': 'No', 'RiverLevel': 'Low', 'Slope': 'Moderate',
              'Season': 'Monsoon', 'Urbanisation': 'High'}
print()
print(f"HAND CALCULATION  = {answer:.4f}   ({answer:.1%})")
print(f"DASHBOARD / pgmpy = {engine.probability('Landslide', 'Yes', coimbatore):.4f}")

soil=Low   :  0.2029 x 0.0190 = 0.0039
soil=Medium:  0.7783 x 0.0900 = 0.0700
soil=High  :  0.0188 x 0.3512 = 0.0066

HAND CALCULATION  = 0.0805   (8.0%)
DASHBOARD / pgmpy = 0.0805


**They match.** The "8.0%" on the dashboard is exactly this arithmetic.

Notice the other readings (temperature, wind, sea temperature, season, land use)
were given to the software but did not change the answer. That is **d-separation**:
they have no open path to "landslide" once rain and humidity are known. The graph
tells us which numbers matter — that is a probabilistic-reasoning fact, not a
learned pattern.

---
## Part 4 — A bigger one: flood, with two hidden variables

Readings: rain **Heavy**, humidity **High**, rural area. Both **soil** and **river**
are unknown, so we must add up all 3 × 3 = 9 combinations.

This is the **chain rule** plus **marginalisation** (summing out what we don't know):

$$P(\text{flood}) = \sum_{\text{soil}} \sum_{\text{river}} P(\text{soil} \mid \text{rain, hum}) \; P(\text{river} \mid \text{rain, soil}) \; P(\text{flood} \mid \text{river, soil, urban})$$

In [9]:
flood = 0.0
soil_high_and_flood = 0.0
for s in STATES["SoilMoisture"]:
    a = table("SoilMoisture").get_value(SoilMoisture=s, Rainfall="Heavy", Humidity="High")
    for r in STATES["RiverLevel"]:
        b = table("RiverLevel").get_value(RiverLevel=r, Rainfall="Heavy", SoilMoisture=s)
        c = table("Flood").get_value(Flood="Yes", RiverLevel=r, SoilMoisture=s, Urbanisation="Low")
        flood += a * b * c
        if s == "High":
            soil_high_and_flood += a * b * c
        print(f"soil={s:<6} river={r:<6}:  {a:.4f} x {b:.4f} x {c:.4f} = {a*b*c:.5f}")

evidence = {"Rainfall": "Heavy", "Humidity": "High", "Urbanisation": "Low"}
print()
print(f"HAND SUM          = {flood:.4f}")
print(f"DASHBOARD / pgmpy = {engine.probability('Flood', 'Yes', evidence):.4f}")

soil=Low    river=Low   :  0.0031 x 0.1419 x 0.0095 = 0.00000
soil=Low    river=Medium:  0.0031 x 0.5038 x 0.0911 = 0.00014
soil=Low    river=High  :  0.0031 x 0.3543 x 0.5000 = 0.00055
soil=Medium river=Low   :  0.0925 x 0.2228 x 0.0708 = 0.00146
soil=Medium river=Medium:  0.0925 x 0.6910 x 0.3032 = 0.01938
soil=Medium river=High  :  0.0925 x 0.0862 x 0.6897 = 0.00550
soil=High   river=Low   :  0.9044 x 0.0509 x 0.1402 = 0.00645
soil=High   river=Medium:  0.9044 x 0.6874 x 0.4936 = 0.30689
soil=High   river=High  :  0.9044 x 0.2617 x 0.8132 = 0.19245

HAND SUM          = 0.5328
DASHBOARD / pgmpy = 0.5328


### Reasoning backwards: *a flood happened — was the soil saturated?*

A machine-learning classifier maps inputs → one output. It cannot be asked this
question without training a new model. The Bayesian network answers it with the
same tables, using Bayes' rule on the rows we just computed:

In [10]:
before = table("SoilMoisture").get_value(SoilMoisture="High", Rainfall="Heavy", Humidity="High")
after  = soil_high_and_flood / flood
print(f"P(soil saturated)                  = {before:.4f}   (before we knew about the flood)")
print(f"P(soil saturated | flood happened) = {soil_high_and_flood:.5f} / {flood:.5f} = {after:.4f}")
print(f"pgmpy                              = {engine.probability('SoilMoisture', 'High', {**evidence, 'Flood': 'Yes'}):.4f}")

P(soil saturated)                  = 0.9044   (before we knew about the flood)
P(soil saturated | flood happened) = 0.50579 / 0.53283 = 0.9493
pgmpy                              = 0.9493


---
## Part 5 — Why this is probabilistic reasoning, not machine learning

| | Machine learning (e.g. random forest) | Our Bayesian network |
|---|---|---|
| What is stored | A fitted function: inputs → output | A **joint probability distribution**, stored as tables via the chain rule |
| How an answer is made | Evaluate the function | **Add and multiply probabilities** (marginalise, apply Bayes' rule) |
| Missing sensor | Must invent a value (imputation) | **Sum over all its possible values** — shown below |
| Reasoning backwards (effect → cause) | Not possible without a new model | Same tables, Bayes' rule — shown above |
| Can you check it by hand? | No | **Yes — this whole notebook** |
| Where numbers come from | Only data | Expert knowledge as a prior, **updated** with data (Dirichlet–multinomial) |

### A missing sensor, handled by probability instead of guessing

If Coimbatore's river gauge breaks, we don't invent a reading. We just **skip Step 2**
and use the first guess about the soil directly:

In [11]:
no_gauge = sum(prior[s] * table("Landslide").get_value(Landslide="Yes", Rainfall="Moderate",
               SoilMoisture=s, Slope="Moderate") for s in STATES["SoilMoisture"])
ev_no_gauge = {k: v for k, v in coimbatore.items() if k != "RiverLevel"}
print(f"with the river gauge    : {answer:.4f}")
print(f"gauge broken (by hand)  : {no_gauge:.4f}")
print(f"gauge broken (pgmpy)    : {engine.probability('Landslide', 'Yes', ev_no_gauge):.4f}")
print()
from src import scenarios
from src.discretize import evidence_from_reading
chittoor = next(r for r in scenarios.build("monsoon_normal") if r["district"] == "Chittoor")
print("On the real dashboard, Chittoor has the same rain, humidity and slope as Coimbatore,")
print(f"but its river gauge IS offline. Dashboard shows: {engine.probability('Landslide', 'Yes', evidence_from_reading(chittoor)):.4f}")

with the river gauge    : 0.0805
gauge broken (by hand)  : 0.0911
gauge broken (pgmpy)    : 0.0911

On the real dashboard, Chittoor has the same rain, humidity and slope as Coimbatore,
but its river gauge IS offline. Dashboard shows: 0.0911


---
## The three formulas to write on the board

**1. Chain rule** — the network stores the joint distribution as a product of small tables:

$$P(x_1, \dots, x_n) = \prod_i P(x_i \mid \text{parents}(x_i))$$

**2. Marginalisation** — for anything we did not measure, add up over all its values:

$$P(A) = \sum_b P(A, B=b)$$

**3. Bayes' rule** — reason from an effect back to a hidden cause:

$$P(\text{cause} \mid \text{effect}) = \frac{P(\text{effect} \mid \text{cause})\,P(\text{cause})}{P(\text{effect})}$$

Every percentage on the dashboard in scenario mode is these three operations applied
to our tables. The pgmpy library only does the same arithmetic faster
(variable elimination = doing the sums in a smart order).